In [1]:
import numpy as np
import pandas as pd
import os
import warnings
import matplotlib.pyplot as plt
from arch import arch_model
import scipy
import statsmodels.api as sm
import importlib
import MarkovAutoregression_t

In [2]:
file_path = 'E:/RA/Geert/task1.py'
file_path = os.path.abspath(file_path)
dir_path = os.path.dirname(file_path)
os.chdir(dir_path)
excel_file = pd.ExcelFile('Aggregate_CPI_inflation_20230513.xls')
sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter['Quarter_str'] = data_quarter['Year'].astype(str) + 'Q' + data_quarter['Quarter'].astype(str)
data_quarter.index = pd.PeriodIndex(data_quarter['Quarter_str'], freq='Q').to_timestamp()
data_month.index = pd.to_datetime(data_month['Year'].astype(str) + data_month['Month'].astype(str).str.zfill(2), format='%Y%m')
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock', 'Quarter_str']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
sample_data = data_quarter[data_quarter['Year'] > 1969].copy()

In [3]:
sample_data['Inflation_lag_1'] = sample_data['Inflation'].shift(1)
sample_data['Inflation_lag_2'] = sample_data['Inflation'].shift(2)
sample_data['Forecasted_inflation_lag_1'] = sample_data['Forecasted inflation'].shift(1)
sample_data = sample_data.dropna()

## Notation

Forecasted inflation--$fc(t-1)$

sigma2--$\sigma^2_{\epsilon (t)}$

ar.L1 -- $\pi(t-1)$

ar.L2 -- $\pi(t-2)$



# Report 

For each mean model(AR,SPF), I report the following model results: 

| Error Distribution | #Regime | Switching AR | Switching SPF | Switching Distribution |
|--------------------|---------|--------------|---------------|------------------------|
| Normal             | 2       | Y            | Y             | Y                      |
| Normal             | 2       | Y            | N             | Y                      |
| Normal             | 2       | N            | N             | Y                      |
| Normal             | 3      | Y            | N             | Y                      |
| Normal             | 3       | N            | N             | Y                      |

In [13]:
def RS_results_autoregression(endog, exog, order):
    switching_variance=True
    switching_nu=True
    trend="n"
    print('\n\n\nRS with Normal distribution, 2 regimes, switching AR, switching SPF and switching variance\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =True,  switching_variance= switching_variance).fit()
    print(res1.summary())
    

    print('\n\n\n\nRS with Normal distribution, 2 regimes, switching AR  and switching variance, without switching SPF\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())
    

    print('\n\n\n\nRS with Normal distribution, 2 regimes,  switching variance, without switching AR and switching SPF\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes= 2,  order=order,  trend=trend, switching_ar=False,
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())
    
    
    print('\n\n\n\nRS with Normal distribution, 3 regimes, switching AR  and switching variance, without switching SPF\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes=3,  order=order,  trend=trend, 
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())

    
    print('\n\n\n\nRS with Normal distribution, 3 regimes,  switching variance, without switching AR and switching SPF\n')
    res1 = sm.tsa.MarkovAutoregression( endog=endog,  exog=exog,  k_regimes= 3,  order=order,  trend=trend, switching_ar=False,
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())
    
def RS_results_regression(endog, exog):
    switching_variance=True
    trend="n"
    print('\n\n\nRS with Normal distribution, 2 regimes and switching variance\n')
    res1 = sm.tsa.MarkovRegression( endog=endog,  exog=exog,  k_regimes= 2,    trend=trend, 
                                       switching_trend=False, switching_exog =True,  switching_variance= switching_variance).fit()
    print(res1.summary())
    

    print('\n\n\n\nRS with Normal distribution, 3 regimes and switching SPF\n')
    res1 = sm.tsa.MarkovRegression( endog=endog,  exog=exog,  k_regimes= 3,    trend=trend, 
                                       switching_trend=False, switching_exog =False,  switching_variance= switching_variance).fit()
    print(res1.summary())

## Empirical Results

$
\pi (t) =  SPF(t-1) + \epsilon (t)
$

I use residuals $\pi(t)-SPF(t-1)$ to estimate RS models. The only parameters are variances and transitional probabilities. $SPF(t-1)$ is  SPF forecast in the time $t-1$ information set.

In [14]:
RS_results_regression(endog= sample_data['Inflation shock'],exog=None)




RS with Normal distribution, 2 regimes and switching variance

                        Markov Switching Model Results                        
Dep. Variable:        Inflation shock   No. Observations:                  210
Model:               MarkovRegression   Log Likelihood                -179.271
Date:                Mon, 25 Dec 2023   AIC                            366.542
Time:                        06:22:26   BIC                            379.930
Sample:                    07-01-1970   HQIC                           371.954
                         - 10-01-2022                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.1403      0.023      6.056      0.000       0.09

$
\pi (t) = c +\rho \pi (t-1)   + \phi fc(t-1) + \epsilon (t)
$

In [5]:
RS_results_autoregression(endog= sample_data['Inflation'],exog=sample_data['Forecasted inflation'], order=1)




RS with Normal distribution, 2 regimes, switching AR, switching SPF and switching variance

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  209
Model:             MarkovAutoregression   Log Likelihood                -172.396
Date:                  Mon, 25 Dec 2023   AIC                            360.792
Time:                          05:30:22   BIC                            387.530
Sample:                      07-01-1970   HQIC                           371.602
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                                  Regime 0 parameters                                   
                           coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------

D:\anaconda\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  209
Model:             MarkovAutoregression   Log Likelihood                -168.241
Date:                  Mon, 25 Dec 2023   AIC                            362.482
Time:                          05:30:25   BIC                            405.932
Sample:                      07-01-1970   HQIC                           380.049
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.1673      0.035      4.760      0.000       0.098       0.236
ar.L1          0.0596      0.080    

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi fc(t-1) + \epsilon (t)
$

In [6]:
RS_results_autoregression(endog= sample_data['Inflation'],exog=sample_data['Forecasted inflation'], order=2)




RS with Normal distribution, 2 regimes, switching AR, switching SPF and switching variance

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  208
Model:             MarkovAutoregression   Log Likelihood                -175.925
Date:                  Mon, 25 Dec 2023   AIC                            371.849
Time:                          05:30:39   BIC                            405.224
Sample:                      07-01-1970   HQIC                           385.344
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                                  Regime 0 parameters                                   
                           coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------

D:\anaconda\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  208
Model:             MarkovAutoregression   Log Likelihood                -157.514
Date:                  Mon, 25 Dec 2023   AIC                            347.027
Time:                          05:30:46   BIC                            400.428
Sample:                      07-01-1970   HQIC                           368.620
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         1.8934      0.768      2.464      0.014       0.387       3.399
ar.L1          0.6742      0.214    

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi_1 fc(t-1) + \phi_2 fc(t-2) + \epsilon (t)
$

In [7]:
RS_results_autoregression(endog= sample_data['Inflation'],exog=sample_data[['Forecasted inflation','Forecasted_inflation_lag_1']], order=2)




RS with Normal distribution, 2 regimes, switching AR, switching SPF and switching variance

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  208
Model:             MarkovAutoregression   Log Likelihood                -172.474
Date:                  Mon, 25 Dec 2023   AIC                            368.949
Time:                          05:31:01   BIC                            408.999
Sample:                      07-01-1970   HQIC                           385.143
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                                     Regime 0 parameters                                      
                                 coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------

                         Markov Switching Model Results                         
Dep. Variable:                Inflation   No. Observations:                  208
Model:             MarkovAutoregression   Log Likelihood                -160.659
Date:                  Mon, 25 Dec 2023   AIC                            355.318
Time:                          05:31:08   BIC                            412.056
Sample:                      07-01-1970   HQIC                           378.260
                           - 10-01-2022                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.9152      0.241      3.791      0.000       0.442       1.388
ar.L1          0.0166      0.057    